In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

## Part 1

In [2]:
conn = sqlite3.connect("data/data.db")

df_original = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
df_reference = pd.read_sql_query("SELECT * FROM ohlc", conn)

df = df_original.copy()

In [3]:
df.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


In [5]:
df.describe(include = "all")

,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,15868,NaN,NaN,NaN,NaN,NaN,NaN,150
top,2020-11-24,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,40,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,NaN,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08,NaN
min,NaN,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,NaN,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,NaN,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,NaN,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN


In [6]:
print("The columns are ", df.columns)
print("\n")
print(
    "ts is timestamp/date\n"
    "open is stock's opening price\n"
    "high is stock's highest daily price\n"
    "low is stock's lowest daily price\n"
    "close is the stock's closing price\n"
    "adj_close is the closing price, adjusted daily for corporate actions\n"
    "volume is the daily number of shares traded for the stock\n"
    "ticker is each stock's ticker symbol")
      

The columns are  Index(['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker'], dtype='str')


ts is timestamp/date
open is stock's opening price
high is stock's highest daily price
low is stock's lowest daily price
close is the stock's closing price
adj_close is the closing price, adjusted daily for corporate actions
volume is the daily number of shares traded for the stock
ticker is each stock's ticker symbol


In [7]:
print("List of tickers ,", df["ticker"].unique())
print("")
print("There are ", df["ticker"].nunique(), "tickers")

List of tickers , <ArrowStringArray>
[                                                                           'CAT',
                                                                            'CVX',
                                                                             'HD',
                                                                           'AAPL',
                                                                            'TRV',
                                                                            'JNJ',
                                                                            'DIS',
                   '<a target="_blank"><span style="color: blue">CSCO</span></a>',
                                                                            'SHW',
                                                                             'KO',
 ...
  '<a target="_blank"><span style="font-weight: bold; color: red">IBM</span></a>',
                    '<a target="_blank"><span

#### Clean ticker column

In [8]:
df["ticker"]=df["ticker"].str.extract(r">([^<]+)<", expand = False).fillna(df["ticker"])
df["ticker"]=df["ticker"].str.strip().str.upper()


In [9]:
print("List of tickers:", df["ticker"].unique())
print("")
print("There are ", df["ticker"].nunique(), "tickers")

List of tickers: <ArrowStringArray>
[ 'CAT',  'CVX',   'HD', 'AAPL',  'TRV',  'JNJ',  'DIS', 'CSCO',  'SHW',
   'KO',   'VZ',  'IBM',   'PG',  'HON',  'AXP',  'MMM',   'GS',   'BA',
  'CRM',  'UNH',  'NKE', 'NVDA', 'AMGN',  'MCD',    'V',  'MRK',  'WMT',
  'JPM', 'MSFT', 'AMZN']
Length: 30, dtype: str

There are  30 tickers


In [10]:
print("Date Range:", df["ts"].min(), "to ", df["ts"].max())

Date Range: 1962-01-02 to  2024-12-06


## 1.2

In [11]:
# function to identify missing values in each column
def detect_missing(df):
    print("Sum of missing values per column:\n")
    return pd.DataFrame(df.isna().sum(),columns = ["Missing values"])




In [12]:
detect_missing(df)

Sum of missing values per column:



,Missing values
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


In [13]:
def detect_missing_dates(frame, print_indicator = True, reference=df_reference):
    missing_rows = []

    for ticker in sorted(frame["ticker"].unique()):
        df = frame.loc[frame["ticker"] == ticker]["ts"]

        longest = df.max()
        shortest = df.min()

        ref = reference.loc[
            (reference["ticker"] == ticker) & (reference["ts"] >= shortest) & (reference["ts"] <= longest)]["ts"]

        missing_dates = ref[~ref.isin(df)]

        if print_indicator == True:
            print( "Ticker", ticker, "is missing", len(missing_dates), "trading days between", shortest, "and", longest)

        for date in missing_dates:missing_rows.append({
                "ticker": ticker,
                "issue_type": "missing trading date",
                "issue_date": date,
                "description": "expected trading date is missing",
                "severity": "medium"
            })

    return pd.DataFrame(missing_rows)

In [14]:
detect_missing_dates(df, True, df_reference)

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

""


#### Test function with fake data

In [15]:
test = df.copy()
test = test.drop([0,50,500])

print("The following tickers should now show missing data:")
print(df.loc[0,"ticker"])
print(df.loc[50,"ticker"])
print(df.loc[500,"ticker"])

The following tickers should now show missing data:
CAT
AMGN
WMT


In [16]:
detect_missing_dates(test, True, df_reference)

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 1 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 1 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

,ticker,issue_type,issue_date,description,severity
0,AMGN,missing trading date,2008-12-05,expected trading date is missing,medium
1,CAT,missing trading date,1975-04-14,expected trading date is missing,medium
2,WMT,missing trading date,2000-10-03,expected trading date is missing,medium


#### Daily Returns

In [17]:
# sort time series column
df = df.sort_values(["ticker", "ts"])

# drop duplicates
rows_before_duplicates = len(df)
df = df.drop_duplicates()
duplicates_removed = rows_before_duplicates - len(df)

#previous close for each stock:
df['prev close'] = df.groupby("ticker")["close"].shift(1)
df['daily_returns'] = ((df['close'] - df['prev close']) / df['prev close'])

#### Outliers and Inconsistencies

In [18]:
def flag_issues(df):

    # create new columns that we will fill in with appropriate data
    df["mean_returns"] = np.nan
    df["std_dev"] = np.nan

    #flag columns to note each issue we find
    df["flag"]=0 #overall flag column
    df["extreme_return_flag"] = 0 #create new flag column, 1's will indicate the column has been flagged
    df["ohlc_flag"]=0
    df["zero_volume_flag"] = 0
    df["invalid_price_flag"] = 0

    # for each ticker, calculate its mean and std, then fill the new columns with teh data
    for ticker in df['ticker'].unique():
        ticker_df = df.loc[df["ticker"]==ticker]
        ticker_mean = ticker_df["daily_returns"].mean()
        ticker_std = ticker_df["daily_returns"].std(ddof=1)

        df.loc[df["ticker"]==ticker, "mean_returns"] = ticker_mean
        df.loc[df["ticker"]==ticker, "std_dev"] = ticker_std

    # flag returns more than 3 standard deviations from the mean
    df.loc[ (df["daily_returns"].notna()) & (df["std_dev"].notna()) & (df["std_dev"] > 0)
        & ((df['daily_returns']-df['mean_returns']).abs()>3*df['std_dev']),['flag',"extreme_return_flag"]]=1

    # flag any impossible data
    df.loc[df["high"]<df["close"],['flag','ohlc_flag']]=1
    df.loc[df["low"]>df["close"],['flag','ohlc_flag']]=1

    #flagging non positive and non finite values
    for column in ["open","high","low","close","adj_close"]:
        df.loc[df[column] <=0,['flag','invalid_price_flag']]=1
        df.loc[~np.isfinite(df[column]),['flag','invalid_price_flag']]=1 # flags NaN as well
        # ~ is 'not'


    #flagging inconsistencies
    df.loc[df['low'] > df['open'],['flag','ohlc_flag']]=1
    df.loc[df['open'] > df['high'],['flag','ohlc_flag']]=1
    df.loc[df['low'] > df['close'],['flag','ohlc_flag']]=1
    df.loc[df['close'] > df['high'],['flag','ohlc_flag']]=1
    df.loc[df['volume']==0,['flag','zero_volume_flag']]=1

    return df

    

In [19]:
df = flag_issues(df)
print("There are",len(df.loc[df['flag']==1]), "flags")

There are 23264 flags


In [20]:
def data_quality_report(df2):
    df = df2.copy()
    extreme_errors_original = df.loc[df['extreme_return_flag']==1]
    extreme_errors = extreme_errors_original[['ticker']].copy()
    extreme_errors['issue_type'] = "outlier return"
    extreme_errors['issue_date'] = extreme_errors_original['ts']
    extreme_errors['description'] = "return exceeds 3 std dev from mean"
    extreme_errors['severity'] = 'medium'

    ohlc_original = df.loc[df['ohlc_flag']==1]
    ohlc_errors = ohlc_original[['ticker']].copy()
    ohlc_errors['issue_type'] = "ohlc error"
    ohlc_errors['issue_date'] = ohlc_original['ts']
    ohlc_errors['description'] = "open or close is outside the daily price bounds"
    ohlc_errors['severity'] = 'high'

    volume_original = df.loc[df['zero_volume_flag']==1]
    volume_errors = volume_original[['ticker']].copy()
    volume_errors['issue_type'] = "trading volume error"
    volume_errors['issue_date'] = volume_original['ts']
    volume_errors['description'] = "zero volume day"
    volume_errors['severity'] = 'low'

    price_original = df.loc[df['invalid_price_flag']==1]
    price_errors = price_original[['ticker']].copy()
    price_errors['issue_type'] = "price error"
    price_errors['issue_date'] = price_original['ts']
    price_errors['description'] = "price is nonpositive or non-finite"
    price_errors['severity'] = 'high'

    # missing dates
    missing_dates = detect_missing_dates(df, False)


    final = pd.concat([extreme_errors, ohlc_errors,volume_errors,price_errors,missing_dates],ignore_index = True)
    
    return final

In [21]:
data_quality_report(df)

,ticker,issue_type,issue_date,description,severity
0,AAPL,outlier return,1980-12-26,return exceeds 3 std dev from mean,medium
1,AAPL,outlier return,1981-04-02,return exceeds 3 std dev from mean,medium
2,AAPL,outlier return,1981-09-25,return exceeds 3 std dev from mean,medium
3,AAPL,outlier return,1982-03-19,return exceeds 3 std dev from mean,medium
4,AAPL,outlier return,1982-11-22,return exceeds 3 std dev from mean,medium
...,...,...,...,...,...
23510,WMT,price error,2024-07-16,price is nonpositive or non-finite,high
23511,WMT,price error,2024-07-29,price is nonpositive or non-finite,high
23512,WMT,price error,2024-09-10,price is nonpositive or non-finite,high
23513,WMT,price error,2024-09-18,price is nonpositive or non-finite,high


## 1.3 Clean the Data


In [22]:
df_clean = df.copy()
df_clean.head()

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag
72412,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,0.001074,0.027846,0,0,0,0,0
102310,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,0.001074,0.027846,0,0,0,0,0
294188,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,0.001074,0.027846,0,0,0,0,0
285108,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,0.001074,0.027846,0,0,0,0,0
194322,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,0.001074,0.027846,0,0,0,0,0


In [23]:
def clean(df_clean, df_reference=df_reference):
    df = df_clean.copy()
    df["repair_flag"] = 0

    rows_before = len(df)

    df = df.dropna(
        subset=["open", "high", "low", "close"],
        how="all"
    ).reset_index(drop=True)

    all_prices_missing_removed = rows_before - len(df)

    df = df.sort_values(["ticker", "ts"])
    
    # Forward fill for each price column
    prices = ["open", "high", "low", "close"]

    # Record which prices were missing before cleaning
    missing_before = df[prices].isna()

    for price in prices:
        df[price] = df.groupby("ticker")[price].ffill()

    # Flag rows where at least one missing price was repaired
    repaired = (missing_before & df[prices].notna()).any(axis=1)

    df.loc[repaired, "repair_flag"] = 1

    #Set missing volume to zero and keep repair flag
    df.loc[df['volume'].isna(),'repair_flag'] = 1
    df.loc[df['volume'].isna(),'volume']=0

    #fix missing adj_close using the reference table
    #calculate adjustment factor using reference table
    reference = df_reference.copy()
    reference["adjustment_factor"] = (reference["adj_close"]/ reference["close"]) # calculate adjustment factor
    # take only needed columns
    reference = reference[["ticker", "ts", "adjustment_factor"]]
    #now merge
    df = df.merge(reference, on=['ts','ticker'],how = 'left') 
    #now need to replace adj_close = NaN values with the new values in "adjustment_factor" column
    # Identify missing adjusted prices
    missing_adj = df["adj_close"].isna()

    # Repair missing adjusted prices
    df.loc[missing_adj, "adj_close"] = (
        df["adjustment_factor"] * df["close"])

    # Flag successfully repaired rows
    repaired_adj = missing_adj & df["adj_close"].notna()
    df.loc[repaired_adj, "repair_flag"] = 1

    #Quarantine unresolved rows
    quarantine= df.loc[df['adj_close'].isna()].copy()
    print("There are", len(quarantine), "unresolved adj_close rows")
    df = df.dropna(subset = ['adj_close']).reset_index(drop=True)
    


    return df, quarantine, all_prices_missing_removed
    

In [24]:
clean_df, quarantine_df, all_prices_missing_removed = clean(df, df_reference)

There are 0 unresolved adj_close rows


In [25]:
# will set any extreme outliers that disagree with reference data equal to NaN, and then run the interpolate function after reviewing all outliers
def handle_outliers(df_clean, df_reference= df_reference):
    df = df_clean.copy()

    # set impossible price relationships to nan and then forward fill
    df.loc[df["close"] > df["high"], "close"] = np.nan
    df.loc[df["close"] < df["low"], "close"] = np.nan
    df.loc[df["open"] > df["high"], "open"] = np.nan
    df.loc[df["open"] < df["low"], "open"] = np.nan
    
    #set any negative or non finite prices to nan and then forward fill
    for column in ["open", "high", "low", "close"]:
        df.loc[df[column] <= 0, column] = np.nan
        df.loc[~np.isfinite(df[column]), column] = np.nan
        
    #forward fill
    df = df.sort_values(["ticker", "ts"])
    prices = ["open", "high", "low", "close"]
    for price in prices:
        df[price] = df.groupby("ticker")[price].ffill()


    #handling extreme returns
        #comparing the extreme returns to that of df_reference, if reference data is the same, then keep the row and the flag
        # if reference data is different, set to Nan and interpolate
    reference = df_reference.copy()

    #calculate returns within the reference table
    reference = reference.sort_values(["ticker", "ts"])
    previous_close = reference.groupby("ticker")["close"].shift(1)
    reference["daily_returns_reference"] = ((reference["close"] - previous_close) / previous_close)
    
    #merge reference table to get df_reference returns
    # take only needed columns
    reference = reference[["ticker", "ts", "daily_returns_reference"]]
    #now merge
    df = df.merge(reference, on=['ts','ticker'],how = 'left') 

    #now look at any row where there is an extreme return that does not equal the reference table's return and set closing price to NaN
    df.loc[(df['extreme_return_flag']==1) & (df['daily_returns']!=df['daily_returns_reference']),'close']=np.nan
    outliers_to_check = df.loc[df['close'].isna()]

    return df, outliers_to_check
    
    

In [26]:
clean_df, outliers_to_check = handle_outliers(clean_df, df_reference)

#### now check if any close prices were set to NaN because of extreme returns

In [27]:
outliers_to_check

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


#### Now finally,  validate that the data is fully clean

In [28]:
# flag clean data again
ohlc_hw_cleaned = flag_issues(clean_df)

#### No missing, non-finite, or nonpositive values in retained raw and adjusted price columns

In [29]:
ohlc_hw_cleaned.loc[ohlc_hw_cleaned['ohlc_flag']==1]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


In [30]:
ohlc_hw_cleaned.loc[ohlc_hw_cleaned['invalid_price_flag']==1]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference
87103,1962-01-02,NaN,3.30000,3.24000,3.30000,0.335750,105840,CVX,NaN,NaN,0.000374,0.016185,1,0,0,0,1,0,0.101742,NaN
167733,1962-01-02,NaN,0.22338,0.22222,0.22338,0.064012,0,JNJ,NaN,NaN,0.000522,0.014459,1,0,0,1,1,0,0.286561,NaN
213412,1966-07-05,NaN,0.27366,0.26749,0.26955,0.118090,388800,MCD,NaN,NaN,0.000678,0.018524,1,0,0,0,1,0,0.438101,NaN
243893,1962-01-02,NaN,0.39703,0.39317,0.39537,0.068447,633830,MRK,NaN,NaN,0.000475,0.015690,1,0,0,0,1,0,0.173121,NaN
287040,1962-01-02,NaN,1.44000,1.42000,1.43000,0.259300,192000,PG,NaN,NaN,0.000396,0.013540,1,0,0,0,1,0,0.181329,NaN


In [31]:
# these are the first available rows for these tickers, forward-fill cannot fix them because there is no earlier open value to carry forward
# solution: drop these rows as they don't have a valid opening price
rows_before = len(ohlc_hw_cleaned)

ohlc_hw_cleaned = ohlc_hw_cleaned.dropna(
    subset=["open"]).reset_index(drop=True)

missing_open_removed = rows_before - len(ohlc_hw_cleaned)
#now final check:
ohlc_hw_cleaned.loc[ohlc_hw_cleaned['invalid_price_flag']==1]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


#### finite non-negative volume check

In [32]:
ohlc_hw_cleaned.loc[ohlc_hw_cleaned['volume']<0]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


In [33]:
ohlc_hw_cleaned.loc[~np.isfinite(ohlc_hw_cleaned['volume'])]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


#### unique (ticker, ts) check

In [34]:
ohlc_hw_cleaned.duplicated(subset=["ticker", "ts"]).sum()

np.int64(0)

#### Verify: All price relationships are valid

In [35]:
ohlc_hw_cleaned.loc[ohlc_hw_cleaned['ohlc_flag']==1]

,ts,open,high,low,close,adj_close,volume,ticker,prev close,daily_returns,mean_returns,std_dev,flag,extreme_return_flag,ohlc_flag,zero_volume_flag,invalid_price_flag,repair_flag,adjustment_factor,daily_returns_reference


### Before / After Statistics

In [36]:
# Row removal breakdown
original_rows = len(df_original)
cleaned_rows = len(ohlc_hw_cleaned)

total_removed = original_rows - cleaned_rows

known_removed = (
    duplicates_removed
    + all_prices_missing_removed
    + len(quarantine_df)
    + missing_open_removed
)

other_removed = total_removed - known_removed

print("Original rows:", original_rows)
print("Cleaned rows:", cleaned_rows)
print("Total removed:", total_removed)

print("Reasons for removal:")
print("Duplicates:", duplicates_removed)
print("All raw prices missing:", all_prices_missing_removed)
print("Unresolved adjusted prices:", len(quarantine_df))
print("Missing opening prices:", missing_open_removed)
print("Other removals:", other_removed)

print("Rows with repair flags:",ohlc_hw_cleaned["repair_flag"].sum())

print("")

print("Before cleaning")
print("missing prices:", df[["open","high","low","close","adj_close"]].isna().sum().sum())
print("invalid price flags:", df["invalid_price_flag"].sum())
print("OHLC flags:", df["ohlc_flag"].sum())

print("")

print("After cleaning")
print("missing prices:", ohlc_hw_cleaned[["open","high","low","close","adj_close"]].isna().sum().sum())
print("invalid price flags:", ohlc_hw_cleaned["invalid_price_flag"].sum())
print("OHLC flags:", ohlc_hw_cleaned["ohlc_flag"].sum())

Original rows: 397711
Cleaned rows: 368246
Total removed: 29465
Reasons for removal:
Duplicates: 29460
All raw prices missing: 0
Unresolved adjusted prices: 0
Missing opening prices: 5
Other removals: 0
Rows with repair flags: 18374

Before cleaning
missing prices: 18374
invalid price flags: 18379
OHLC flags: 5

After cleaning
missing prices: 0
invalid price flags: 0
OHLC flags: 0


In [38]:
# validation
print("Total repaired rows:",
      (ohlc_hw_cleaned["repair_flag"] == 1).sum())

print("Unflagged repairs:",
      (ohlc_hw_cleaned["repair_flag"].isna()).sum())

Total repaired rows: 18374
Unflagged repairs: 0


#### Drop helper columns

In [39]:
ohlc_hw_cleaned = ohlc_hw_cleaned[['ts','open','high','low','close','adj_close','volume','ticker', 'repair_flag']]

#### Store ts consistently as ISO dates and upload

In [40]:
ohlc_hw_cleaned["ts"] = pd.to_datetime(ohlc_hw_cleaned["ts"]).dt.strftime("%Y-%m-%d")

In [41]:
ohlc_hw_cleaned.to_sql("ohlc_hw_cleaned",conn, if_exists="replace",index=False)
conn.close()

In [42]:
ohlc_hw_cleaned

,ts,open,high,low,close,adj_close,volume,ticker,repair_flag
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,0
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0
...,...,...,...,...,...,...,...,...,...
368241,2024-10-14,80.05000,80.51000,79.76000,80.29000,80.290000,9914500,WMT,0
368242,2024-10-15,80.67000,81.82000,80.51000,81.65000,81.650000,13276221,WMT,0
368243,2024-10-16,81.53000,81.66000,80.62000,81.22000,81.220000,11123300,WMT,0
368244,2024-10-17,81.27000,81.50000,80.65000,80.89000,80.890000,9424705,WMT,0


### Extra data cleaning
While completing Part 2, I noticed unusually high volatility and drawdown values caused by incorrect adjusted closing prices. I added this additional cleaning step to identify suspicious adjusted prices and replace them with values from the reference dataset when available. 
After adding this step, volatility and drawdown values returned to more reasonable figures.

In [43]:
# Identify suspicious adjusted closing prices
suspicious = (
    (ohlc_hw_cleaned["adj_close"] > 3 * ohlc_hw_cleaned["close"]) |
    (ohlc_hw_cleaned["adj_close"] <= 0)
)

# Match reference prices by ticker and date
reference_prices = df_reference[
    ["ticker", "ts", "adj_close"]
].rename(columns={"adj_close": "reference_adj_close"})

ohlc_hw_cleaned = ohlc_hw_cleaned.merge(
    reference_prices,
    on=["ticker", "ts"],
    how="left"
)

# Replace suspicious values where reference data is available
repair = suspicious.to_numpy() & ohlc_hw_cleaned["reference_adj_close"].notna()

ohlc_hw_cleaned.loc[repair, "adj_close"] = (
    ohlc_hw_cleaned.loc[repair, "reference_adj_close"]
)

ohlc_hw_cleaned.loc[repair, "repair_flag"] = 1

# Remove temporary reference column
ohlc_hw_cleaned = ohlc_hw_cleaned.drop(columns="reference_adj_close")

print("Adjusted prices repaired:", repair.sum())

Adjusted prices repaired: 9126


In [44]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)

conn = sqlite3.connect("data/data.db")

ohlc_hw_cleaned.to_sql(
    "ohlc_hw_cleaned",
    conn,
    if_exists="replace",
    index=False
)

conn.close()